In [ ]:
import cv2 as cv
import numpy as np
import pandas as pd
from nmot import NMOT

In [ ]:
def process_video(
    input_path: str,
    output_path: str = "tracked_output.mp4",
    csv_path: str = "tracks.csv",
    roi=None,
):
    cap = cv.VideoCapture(input_path)

    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {input_path}")

    fps = cap.get(cv.CAP_PROP_FPS)
    width = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv.VideoWriter_fourcc(*"mp4v")
    writer = cv.VideoWriter(output_path, fourcc, fps, (width, height))

    tracker = NMOT(
        warmup_frames=50,
        min_area=8,
        max_area=80,
        max_match_dist=25,
        max_missed=12,
        use_lk=True,
        roi=roi,
    )

    while True:
        ok, frame = cap.read()

        if not ok:
            break

        vis, mask, active_tracks = tracker.update(frame)
        writer.write(vis)

        
        cv.imshow("tracks", vis)
        cv.imshow("mask", mask)
        if cv.waitKey(1) & 0xFF == 27:
            break

    cap.release()
    writer.release()
    cv.destroyAllWindows()

    df = tracker.save_tracks(csv_path)

    return df

In [ ]:
df = process_video(
    input_path=r"data\S2170006.MP4",
    output_path="ants_tracked.mp4",
    csv_path="ants_tracks.csv",
    roi=None,
)

print(df.head())